# 08 — Incremental reaction prediction
This experiment tests the count model discussed in the conversation, rather than changing notebook 07's bands. History is stored as rejection/break counts and separate duration sums. Each prediction is saved before its outcome enters the model.

**Fixed experiment:** same 50,000 canonical AAPL trades; first 25,000 events warm up; evaluate encounters starting afterward while continuing to learn. Support and resistance have separate shared tables; age at touch is under 60 seconds or at least 60 seconds. Shared history carries weight 4. These are illustrative, preselected settings, not optimized parameters.

**Band outcome:** support rejects on reaching upper boundary + $0.02, resistance on reaching lower boundary − $0.02; strict crossing of the opposite boundary is a break, matching notebook 07. This extends the point-level example to bands and preserves the existing break rule instead of imposing a new one-cent break buffer. No outcome timeout is imposed; unresolved encounters are reported separately. Accuracy describes these small-distance outcomes, not major swings or trading profitability.

In [ ]:
import os, sys, json, time, hashlib, subprocess
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display
REPO = Path('D:/TradingCodes/quant-research-workbench')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from research.level_book.v8.pressure_engine import run_fast
from research.level_book.v8.incremental_prediction import encounters, CountModel, predict_online, metrics
ROOT = Path('D:/TradingML/runtimes')
assert ROOT.is_dir(), 'Required runtime root unavailable'
RUN = ROOT/'level-book-v8/incremental-prediction'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir(parents=True, exist_ok=False)
SOURCE = ROOT/'level-book-v8/pressure-test/source-50000'
CUTOFF = 25000
DISTANCE = 200  # $0.02 in the canonical integer price representation (1 unit = $0.0001).
SESSION_OPEN_US = 1787232600000000  # 2026-08-20 09:30 America/New_York, expressed in UTC microseconds.
print('Runtime output:', RUN)

## 1. Verify and inspect the same source
Read only the existing canonical-derived cache and verify its provenance hash. There is no raw-file or network fallback. Order is timestamp followed by ordinal; event is the one-based position in that order. Equal timestamps remain distinct events. Durations may legitimately be zero.

In [ ]:
source = pl.read_parquet(SOURCE/'source_trades.parquet')
provenance = json.loads((SOURCE/'source_provenance.json').read_text(encoding='utf-8'))
records = source.select('ordinal', 'time_us', 'price_units', 'volume').to_dicts()
for row in records:
    if float(row['volume']).is_integer():
        row['volume'] = int(row['volume'])
source_hash = hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance['sample_sha256']
assert source.height == 50000 and provenance['ticker'] == 'AAPL' and provenance['day'] == '2026-08-20'
assert source['ordinal'].n_unique() == source.height
assert source.select('time_us','ordinal').equals(source.sort('time_us','ordinal').select('time_us','ordinal'))
del records
print('Events:', source.height, '| split event:', CUTOFF)
display(source.head(10))

## 2. Reuse the causal, stable bands and define encounters
Run notebook 07's engine unchanged. A prediction requires a fresh entry into an active band **from its approach side after activation**. Support approaches from above; resistance from below. Activation-time touches are excluded because they already participate in band discovery. Jumping across the entire band is not a touch. Repeated prints inside a band do not create more encounters. Only one encounter can be pending per role period.

The extractor uses vectorized price comparisons within each period and a small loop over encounters to enforce this rule. It reads subsequent prices to assign evaluation labels, but those columns are withheld from prediction and only admitted after resolution. A role period resets personal evidence on a role change; shared history retains prior periods.

In [ ]:
started = time.perf_counter()
replay = run_fast(source, SESSION_OPEN_US)
encounter_table = encounters(replay, DISTANCE)
print('Band replay + encounter extraction seconds:', round(time.perf_counter()-started, 3))
print('Bands:', len(replay.lower), '| role periods:', len(replay.periods), '| eligible encounters:', encounter_table.height)
display(encounter_table.head(15))
display(encounter_table.group_by('role','age_group').agg(pl.len().alias('encounters'), pl.col('outcome').is_null().sum().alias('pending')))
encounter_table.write_parquet(RUN/'encounters.parquet')

## 3. The stored model and update rule
Each `(role, age group)` stores `[rejections, breaks, rejection_seconds_sum, break_seconds_sum]`. Each role period stores the same aggregates, split by age-at-touch group. Pending encounters retain their original group even if the level later ages past 60 seconds.

For a prediction, subtract this period's own group counts from the shared counts, avoiding double counting. Let shared counts be R and B, and personal counts across both ages be r and b:

`group_probability = (R + 1) / (R + B + 2)`

`prediction = (r + 4 * group_probability) / (r + b + 4)`

A new level gets the shared estimate immediately. An older level benefits from its personal evidence, not an automatic age bonus. Reaction duration is stored and reported; it does not yet change probability during a pending encounter. Age uses two coarse groups, so this is a deliberately simple baseline, not a final granular model.

The executable implementation is in `incremental_prediction.py`; the example below verifies the arithmetic independently.

In [ ]:
toy = CountModel()
# Six completed resistance encounters: three young, three established.
for pid, group, outcome, seconds in [(0,0,1,2),(1,0,1,4),(2,0,0,1),(3,1,1,3),(4,1,0,2),(5,1,0,4)]:
    toy.update(dict(period_id=pid,role=1,age_group=group,outcome=outcome,duration_seconds=seconds))
assert np.isclose(toy.predict(10,1,0)['prediction'], .6)
assert np.isclose(toy.predict(11,1,1)['prediction'], .4)
toy.update(dict(period_id=10,role=1,age_group=0,outcome=1,duration_seconds=3))
assert np.isclose(toy.predict(10,1,0)['prediction'], .68)
assert np.isclose(toy.predict(12,1,0)['prediction'], 2/3)
display(pl.DataFrame([dict(level='G after rejection', **toy.predict(10,1,0)), dict(level='New young level', **toy.predict(12,1,0))]))
print('Stored resistance aggregates [R, B, seconds_R, seconds_B]:', toy.groups[1])

## 4. Predict first, then update when outcomes become available
The learner processes touches in event order. Before a touch at event e, it incorporates only outcomes with resolution event strictly less than e. Simultaneous touches therefore see the same history. Warmup encounters that resolve after the split are learned only when they resolve.

Compare the age-conditioned personal model against its shared-only estimate, an age-free online role baseline, an online global rejection-rate baseline, and a frozen warmup rejection-rate baseline. All predictions are recorded at touch; future labels cannot modify saved probabilities.

In [ ]:
predictions, model = predict_online(encounter_table, CUTOFF, source.height)
warm_resolved = encounter_table.filter(pl.col('resolve_event') <= CUTOFF)
frozen_probability = (warm_resolved['outcome'].sum()+1)/(warm_resolved.height+2)
predictions = predictions.with_columns(pl.lit(frozen_probability).alias('frozen_prediction'))
assert predictions.filter(pl.col('latest_training_event') >= pl.col('touch_event')).is_empty()
assert predictions.filter(pl.col('resolve_event').is_not_null() & (pl.col('resolve_event') <= pl.col('touch_event'))).is_empty()
display(predictions.filter(pl.col('evaluation')).select('touch_event','period_id','role','age_seconds','personal_rejections','shared_rejections','shared_breaks','prediction','global_prediction','resolve_event','outcome').head(20))
predictions.write_parquet(RUN/'predictions.parquet')

## 5. Evaluate the second half without tuning on it
Accuracy uses probability >= 0.5 as rejection. Balanced accuracy gives rejection and break equal weight. ROC AUC measures ranking (0.5 means chance ranking). Brier score and log loss assess probabilities (lower is better), avoiding a misleading accuracy win from the majority class. Pending outcomes are not failures and are excluded from these metrics; their count remains visible.

This is one ticker/session, and overlapping bands can generate correlated encounters. These are descriptive out-of-time results within this sample, not independent-trial significance or cross-session validation.

In [ ]:
evaluation = predictions.filter(pl.col('evaluation'))
resolved = evaluation.filter(pl.col('outcome').is_not_null())
assert resolved.height and resolved['outcome'].n_unique() == 2
columns = ['prediction','group_prediction','role_prediction','global_prediction','frozen_prediction']
summary = pl.DataFrame([metrics(resolved,c) for c in columns])
coverage = dict(evaluation_encounters=evaluation.height, resolved=resolved.height,
                pending=evaluation.height-resolved.height, warmup_encounters=predictions.height-evaluation.height)
print('Coverage:', coverage)
display(summary)
subgroups = []
for (role,age), group in resolved.group_by('role','age_group'):
    if group['outcome'].n_unique() == 2:
        subgroups.append(dict(role=role,age_group=age,**metrics(group,'prediction')))
display(pl.DataFrame(subgroups))
summary.write_parquet(RUN/'metrics.parquet')
# Fixed bins expose both discrimination and miscalibration without fitting anything.
calibration = resolved.with_columns((pl.col('prediction')*10).floor().clip(0,9).cast(pl.Int64).alias('bin')).group_by('bin').agg(
    pl.len().alias('count'),pl.col('prediction').mean().alias('mean_prediction'),pl.col('outcome').mean().alias('observed_rejection')).sort('bin')
display(calibration)
summary.write_parquet(RUN/'metrics.parquet')
calibration.write_parquet(RUN/'calibration.parquet')
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].plot([0,1],[0,1],color='gray',ls='--')
axes[0].scatter(calibration['mean_prediction'],calibration['observed_rejection'],s=np.sqrt(calibration['count'].to_numpy())*12)
axes[0].set(xlabel='Predicted rejection probability',ylabel='Observed rejection fraction',xlim=(0,1),ylim=(0,1),title='Calibration; marker size reflects count')
axes[1].bar(summary['model'].to_list(),summary['brier'].to_list())
axes[1].tick_params(axis='x',rotation=25)
axes[1].set(ylabel='Brier score (lower is better)',title='Same resolved evaluation encounters')
fig.tight_layout(); fig.savefig(RUN/'evaluation.png',dpi=130); plt.show(); plt.close(fig)

## 6. Check chronology and label semantics
Replaying a shorter price prefix must reproduce all earlier predictions. Truncating a pending encounter must leave it pending, never convert it to a failure. A hand-built resistance path verifies rejection, a later re-entry and break. These tests exercise the executable extractor and learner, not just the displayed formulas.

In [ ]:
from types import SimpleNamespace
# Price units below are deliberately tiny to make the boundary arithmetic readable.
fake_prices = np.array([99,100,99,100,98,99,100,101])
fake = SimpleNamespace(frame=pl.DataFrame({'price_units':fake_prices}),times=np.arange(8)*1000000,
    periods=np.array([[0,1,1,2,-1]]),ends=np.array([8]),values=np.array([100]),lower=np.array([0]),upper=np.array([0]))
small = encounters(fake,2)
assert small['touch_event'].to_list() == [4,7]
assert small['resolve_event'].to_list() == [5,8]
assert small['outcome'].to_list() == [1,0]
# Mirroring prices and role must preserve the encounter outcomes.
mirror = SimpleNamespace(**vars(fake))
mirror.frame = pl.DataFrame({'price_units':200-fake_prices})
mirror.periods = np.array([[0,0,1,2,-1]])
assert encounters(mirror,2)['outcome'].to_list() == [1,0]
# A resolution on the same event as another touch is not used for that touch.
tied = pl.DataFrame([dict(period_id=0,role=1,age_group=0,touch_event=2,resolve_event=4,outcome=1,duration_seconds=2.),
                     dict(period_id=1,role=1,age_group=0,touch_event=4,resolve_event=6,outcome=0,duration_seconds=2.),
                     dict(period_id=2,role=1,age_group=0,touch_event=5,resolve_event=None,outcome=None,duration_seconds=None)])
tied_pred, _ = predict_online(tied,3,6)
assert tied_pred['prediction'].to_list() == [.5,.5,2/3]
# Truncate before the rejection: the first encounter is unresolved.
fake.frame = fake.frame.head(4); fake.times = fake.times[:4]; fake.ends = np.array([0])
assert encounters(fake,2)['outcome'].to_list() == [None]
prefix_replay = run_fast(source.head(CUTOFF),SESSION_OPEN_US)
prefix_table = encounters(prefix_replay,DISTANCE)
prefix_predictions, _ = predict_online(prefix_table,CUTOFF,CUTOFF)
check_columns = ['period_id','touch_event','prediction','group_prediction','global_prediction','latest_training_event']
assert prefix_predictions.select(check_columns).equals(predictions.filter(pl.col('touch_event') <= CUTOFF).select(check_columns))
# Every outcome contributes exactly once to the shared model.
completed = encounter_table.filter(pl.col('resolve_event').is_not_null())
assert int(model.total[:2].sum()) == completed.height
assert int(model.total[0]) == completed['outcome'].sum()
print('PASS: toy arithmetic, rejection/break/pending labels, prefix predictions, strict training chronology, one update per outcome')

## 7. Inspect individual predictions interactively
Choose an event to inspect the most recent encounters known by that event. Future outcomes are hidden; the probability shown is the prediction saved at touch. This viewer does not rerun the model or retrospectively change predictions. The table exposes new versus experienced levels and the history behind each score.

In [ ]:
import ipywidgets as widgets
view_state = {}
def inspect(event):
    visible = predictions.filter(pl.col('touch_event') <= event).tail(20).with_columns(
        pl.when(pl.col('resolve_event') <= event).then(pl.col('outcome')).otherwise(None).alias('known_outcome'),
        pl.when(pl.col('resolve_event') <= event).then(pl.col('duration_seconds')).otherwise(None).alias('known_duration'))
    display(visible.select('touch_event','period_id','role','age_seconds','prediction','personal_rejections','shared_rejections','shared_breaks','mean_rejection_seconds','known_outcome','known_duration'))
    view_state.update(event=event,rows=visible.height)
slider = widgets.IntSlider(value=source.height,min=200,max=source.height,description='As-of event',continuous_update=False,layout=widgets.Layout(width='850px'))
viewer = widgets.interactive_output(inspect,{'event':slider})
display(widgets.VBox([slider,viewer]))

## 8. Save model state and evidence
Store the aggregate tables, unresolved encounters and last source ordinal/time together. These are inspectable research snapshots; production streaming restore also requires the band engine state and atomic source-offset handling, which this notebook does not implement. No database writes occur.

Next decisions should follow the evaluation: whether age groups help, whether personal evidence improves on shared history, and whether the two-cent target captures the intended reaction. Do not select new settings on this evaluation sample and call the result held out.

In [ ]:
group_rows = [dict(role=r,age_group=g,rejections=int(model.groups[r,g,0]),breaks=int(model.groups[r,g,1]),rejection_seconds=model.groups[r,g,2],break_seconds=model.groups[r,g,3]) for r in range(2) for g in range(2)]
group_state = pl.DataFrame(group_rows)
display(group_state)
group_state.write_parquet(RUN/'shared_model.parquet')
personal_rows = [dict(period_id=pid,age_group=g,rejections=int(a[g,0]),breaks=int(a[g,1]),rejection_seconds=a[g,2],break_seconds=a[g,3]) for pid,a in model.personal.items() for g in range(2)]
pl.DataFrame(personal_rows).write_parquet(RUN/'personal_model.parquet')
predictions.filter(pl.col('resolve_event').is_null()).write_parquet(RUN/'pending.parquet')
manifest = dict(method='08_incremental_reaction_prediction',git_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(),
    source_sha256=source_hash,source=str(SOURCE),run=str(RUN),events=source.height,cutoff=CUTOFF,
    distance_units=DISTANCE,age_boundary_seconds=60,shared_weight=4,last_ordinal=int(source['ordinal'][-1]),last_time_us=int(source['time_us'][-1]),
    coverage=coverage,metrics=summary.to_dicts(),validation='passed',
    notebook_sha256=hashlib.sha256((REPO/'research/level_book/v8/08_incremental_reaction_prediction.ipynb').read_bytes()).hexdigest(),
    implementation_sha256=hashlib.sha256((REPO/'research/level_book/v8/incremental_prediction.py').read_bytes()).hexdigest())
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Evidence saved:',RUN)
print(summary)

## 9. Read the result before changing the model
Compare Brier error first: a useful probability model should improve on simple historical frequencies. The role-only baseline helps isolate the value of splitting by age, although the shared-only model also excludes the current period. The personal blend can overestimate rejection: active role periods have only successes so far, because their first break terminates the period. Repeated successes do not necessarily imply a higher chance on the next encounter.
A positive result on one session would still need new sessions/tickers. A negative result is evidence against this particular score, not against every possible incremental model. Keep these settings fixed for this evaluation; duration-aware prediction and richer approach features are separate experiments.


In [ ]:
model_row = summary.filter(pl.col('model') == 'prediction').row(0,named=True)
baseline_row = summary.filter(pl.col('model') == 'global_prediction').row(0,named=True)
print(f"Model accuracy: {model_row['accuracy']:.1%}; balanced accuracy: {model_row['balanced_accuracy']:.1%}")
print(f"Model Brier: {model_row['brier']:.4f}; online global baseline: {baseline_row['brier']:.4f}")
print('Probability error improves on global baseline.' if model_row['brier'] < baseline_row['brier'] else 'Probability error does NOT improve on global baseline; do not promote this score.')
print('Mean predicted rejection:', round(resolved['prediction'].mean(),4), '| observed:', round(resolved['outcome'].mean(),4))
